# 02 — Feature Engineering with WOE and IV

Bin boundaries and WOE mappings are fitted on training data only.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import seaborn as sns
from src.scorecard import DEFAULT_FEATURES, fit_woe_bins, load_creditcard_csv, split_data

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'creditcard.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURE_DIR = PROJECT_ROOT / 'figures'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = load_creditcard_csv(DATA_PATH)
X_train, X_test, y_train, y_test = split_data(df, test_size=0.30, random_state=42)
print('Train:', X_train.shape, 'Fraud rate:', round(y_train.mean(), 6))
print('Test :', X_test.shape, 'Fraud rate:', round(y_test.mean(), 6))

In [ ]:
candidate_features = [f'V{i}' for i in range(1, 29)] + ['Amount', 'Time']
woe_definitions, iv_table = fit_woe_bins(
    X_train, y_train, candidate_features, n_bins=10, alpha=0.5
)
iv_table.head(15)

In [ ]:
ax = sns.barplot(data=iv_table.head(15), x='iv', y='feature', color='#4C78A8')
ax.set(title='Top Features by Information Value', xlabel='Information Value', ylabel='Feature')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'iv_ranking.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
selected_features = DEFAULT_FEATURES
iv_table.to_csv(PROCESSED_DIR / 'iv_summary.csv', index=False)
(PROCESSED_DIR / 'selected_features.json').write_text(
    json.dumps(selected_features, indent=2), encoding='utf-8'
)
selected_features

The final ten variables are kept consistent with the completed project analysis. Notebook 03 refits their WOE mappings on its training sample and stores them with the model.